In [1]:
import json
import cv2 as cv
from pathlib import Path

# Paths
base_dir = Path.cwd()  # current working directory
data_dir = base_dir / "labeled_example_data"
labels_path = data_dir / "labels.json"
output_dir = base_dir / "labeled_with_red_dot"
output_dir.mkdir(exist_ok=True)

# Load labels
with open(labels_path, "r") as f:
    labels = json.load(f)

# Loop through all entries
for entry in labels:
    img_path = data_dir / entry["filename"]
    img = cv.imread(str(img_path))
    if img is None:
        print(f"⚠️ Could not read {img_path}")
        continue

    h, w = img.shape[:2]

    # Draw red dot if wheel is present
    if entry["is_present"] == 1.0:
        x_px = int(entry["center"][0] * w)
        y_px = int(entry["center"][1] * h)
        cv.circle(img, (x_px, y_px), 5, (0, 0, 255), -1)  # red filled circle

    # Save annotated image
    out_path = output_dir / entry["filename"]
    cv.imwrite(str(out_path), img)

print(f"✅ Annotated images saved to: {output_dir}")

✅ Annotated images saved to: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/labeled_with_red_dot


In [2]:
import json
import cv2 as cv
import numpy as np
from pathlib import Path
import csv

# --- Paths ---
base_dir    = Path.cwd()
data_dir    = base_dir / "labeled_example_data"
labels_path = data_dir / "labels.json"
output_dir  = base_dir / "debug_check"
output_dir.mkdir(exist_ok=True)
csv_path    = base_dir / "detection_vs_gt.csv"

# --- Load labels ---
with open(labels_path) as f:
    labels = json.load(f)
labels_by_name = {e["filename"]: e for e in labels}

# --- Helpers ---
def crop_around_center(img, cx, cy, size=320):
    h, w = img.shape[:2]
    half = size // 2
    x0 = max(0, cx-half);  x1 = min(w, cx+half)
    y0 = max(0, cy-half);  y1 = min(h, cy+half)
    return img[y0:y1, x0:x1], x0, y0

def estimate_radius(edges, cx, cy):
    ys, xs = np.nonzero(edges)
    if len(xs)==0: return None
    rs = np.hypot(xs-cx, ys-cy)
    rs = rs[(rs>5)&(rs<min(cx,cy)*0.9)]
    if len(rs)==0: return None
    hist, bins = np.histogram(rs, bins=int(min(cx,cy)))
    return bins[np.argmax(hist)]

def detect_center_hough_constrained(gray, r_hint, tol=0.2):
    h, w = gray.shape[:2]
    blur = cv.GaussianBlur(gray, (9,9), 2)
    minR = max(5, int(r_hint*(1-tol)))
    maxR = max(minR+1, int(r_hint*(1+tol)))
    circles = cv.HoughCircles(
        blur, cv.HOUGH_GRADIENT,
        dp=1.2, minDist=r_hint*0.5,
        param1=100, param2=25,
        minRadius=minR, maxRadius=maxR
    )
    if circles is None:
        return None
    # pick the circle whose center is closest to image center
    cx0, cy0 = w//2, h//2
    cands = circles[0]
    best = min(cands, key=lambda c: (c[0]-cx0)**2 + (c[1]-cy0)**2)
    x, y, r = best
    return int(round(x)), int(round(y)), r

# --- Batch processing ---
errors = []
rows   = []
for fname, entry in labels_by_name.items():
    img_path = data_dir / fname
    img = cv.imread(str(img_path))
    if img is None or entry["is_present"]!=1.0:
        continue

    H, W = img.shape[:2]
    # Ground-truth center in pixels
    gt_x = int(round(entry["center"][0]*W))
    gt_y = int(round(entry["center"][1]*H))

    # Crop around GT
    crop, x0, y0 = crop_around_center(img, gt_x, gt_y)
    gray_crop = cv.cvtColor(crop, cv.COLOR_BGR2GRAY)
    edges = cv.Canny(gray_crop, 50, 150)
    
    # Estimate radius and detect
    r_exp = estimate_radius(edges, W//2 - x0, H//2 - y0)
    det = None
    if r_exp:
        det = detect_center_hough_constrained(gray_crop, r_exp, tol=0.2)
    if not det:
        # final fallback to full-image Hough
        det = detect_center_hough_constrained(cv.cvtColor(img, cv.COLOR_BGR2GRAY), r_exp or min(H,W)//4, tol=0.5)
    if det:
        dx, dy, _ = det
        det_x = dx + x0;  det_y = dy + y0
        err = float(np.hypot(det_x-gt_x, det_y-gt_y))
        errors.append(err)
        suspect = int(err>15)
    else:
        det_x = det_y = None
        err = None
        suspect = 1

    # Draw overlay
    cv.circle(img, (gt_x, gt_y), 6, (0,0,255), -1)
    cv.drawMarker(img, (gt_x, gt_y), (0,0,255),
                  markerType=cv.MARKER_CROSS, markerSize=16, thickness=2)
    if det_x and det_y:
        cv.circle(img, (det_x, det_y), 6, (0,255,0), 2)
        cv.drawMarker(img, (det_x, det_y), (0,255,0),
                      markerType=cv.MARKER_TILTED_CROSS, markerSize=20, thickness=2)

    # Save and log
    cv.imwrite(str(output_dir/fname), img)
    rows.append([fname, gt_x, gt_y, det_x, det_y, err, suspect])

# --- Write CSV ---
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filename","gt_x","gt_y","det_x","det_y","error_px","suspect"])
    writer.writerows(rows)

# --- Summary ---
if errors:
    print(f"🔍 {len(errors)} detections | avg error {np.mean(errors):.2f}px | median {np.median(errors):.2f}px")
print(f"⚠️ Overlays: {output_dir}\n📄 Report: {csv_path}")

🔍 907 detections | avg error 33.43px | median 30.87px
⚠️ Overlays: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/debug_check
📄 Report: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/detection_vs_gt.csv


#  Visualize Predictions

In [3]:
import json
import cv2 as cv
from pathlib import Path

# --- Paths ---
base_dir    = Path.cwd()                     # project root
data_dir    = base_dir / "labeled_example_data"
labels_path = data_dir / "labels.json"
output_dir  = base_dir / "debug_check_manual"
output_dir.mkdir(exist_ok=True)

# --- Manual normalized-center overrides ---
# Specify filename → (x_norm, y_norm)
manual_centers_norm = {
    "rim_000.png": (0.30, 0.89),
    # Add more overrides here if you like, e.g.:
    # "rim_030.png": (0.55, 0.61),
}

# --- Load and patch labels ---
with open(labels_path, "r") as f:
    labels = json.load(f)

for entry in labels:
    fname = entry["filename"]
    if fname in manual_centers_norm:
        entry["center"] = list(manual_centers_norm[fname])
        print(f"Overrode center for {fname}: {entry['center']}")

# --- Draw red GT circle using (possibly overridden) centers ---
for entry in labels:
    fname = entry["filename"]
    img_path = data_dir / fname

    # Skip missing or non-present images
    if not img_path.exists():
        print(f"⚠️ Missing file: {fname}")
        continue
    if entry.get("is_present", 0.0) != 1.0:
        continue

    # Load image
    img = cv.imread(str(img_path))
    if img is None:
        print(f"⚠️ Could not read: {fname}")
        continue

    H, W = img.shape[:2]

    # Convert normalized → pixel coords
    x_norm, y_norm = entry["center"]
    gt_x = int(round(x_norm * W))
    gt_y = int(round(y_norm * H))

    # Draw red circle + crosshair
    cv.circle(    img, (gt_x, gt_y),  6, (0, 0, 255), -1)
    cv.drawMarker(img, (gt_x, gt_y), (0, 0, 255),
                  markerType=cv.MARKER_CROSS,
                  markerSize=16, thickness=2)

    # Save annotated image
    out_path = output_dir / fname
    cv.imwrite(str(out_path), img)

print(f"✅ Overlays with manual GT centers saved to: {output_dir}")

Overrode center for rim_000.png: [0.3, 0.89]
✅ Overlays with manual GT centers saved to: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/debug_check_manual


In [4]:
import json
import cv2 as cv
from pathlib import Path

# --- Paths ---
base_dir    = Path.cwd()                     # project root
data_dir    = base_dir / "labeled_example_data"
labels_path = data_dir / "labels.json"
output_dir  = base_dir / "debug_check_manual"
output_dir.mkdir(exist_ok=True)

# --- Manual normalized-center overrides ---
# Specify filename → (x_norm, y_norm)
manual_centers_norm = {
    "rim_001.png": (0.30, 0.89),
    # Add more overrides here if you like, e.g.:
    # "rim_030.png": (0.55, 0.61),
}

# --- Load and patch labels ---
with open(labels_path, "r") as f:
    labels = json.load(f)

for entry in labels:
    fname = entry["filename"]
    if fname in manual_centers_norm:
        entry["center"] = list(manual_centers_norm[fname])
        print(f"Overrode center for {fname}: {entry['center']}")

# --- Draw red GT circle using (possibly overridden) centers ---
for entry in labels:
    fname = entry["filename"]
    img_path = data_dir / fname

    # Skip missing or non-present images
    if not img_path.exists():
        print(f"⚠️ Missing file: {fname}")
        continue
    if entry.get("is_present", 0.0) != 1.0:
        continue

    # Load image
    img = cv.imread(str(img_path))
    if img is None:
        print(f"⚠️ Could not read: {fname}")
        continue

    H, W = img.shape[:2]

    # Convert normalized → pixel coords
    x_norm, y_norm = entry["center"]
    gt_x = int(round(x_norm * W))
    gt_y = int(round(y_norm * H))

    # Draw red circle + crosshair
    cv.circle(    img, (gt_x, gt_y),  6, (0, 0, 255), -1)
    cv.drawMarker(img, (gt_x, gt_y), (0, 255, 0),
                  markerType=cv.MARKER_CROSS,
                  markerSize=16, thickness=2)

    # Save annotated image
    out_path = output_dir / fname
    cv.imwrite(str(out_path), img)

print(f"✅ Overlays with manual GT centers saved to: {output_dir}")

Overrode center for rim_001.png: [0.3, 0.89]
✅ Overlays with manual GT centers saved to: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/debug_check_manual


In [5]:
import json
import cv2 as cv
from pathlib import Path

# --- Paths ---
base_dir    = Path.cwd()                     # project root
data_dir    = base_dir / "labeled_example_data"
labels_path = data_dir / "labels.json"
output_dir  = base_dir / "debug_check_manual"
output_dir.mkdir(exist_ok=True)

manual_centers_norm = {
    "rim_001.png": (0.30, 0.89),
    # Add more overrides here if you like, e.g.:
    # "rim_030.png": (0.55, 0.61),
}


img = cv.imread(str(data_dir / "rim_001.png"))
H, W = 128, 128
cv.circle(img, (64, 64), 5, (255, 0, 255), -1)

out_path = output_dir / "rim_002.png"
cv.imwrite(str(out_path), img)



True

In [6]:
print("data_dir: ", data_dir)

data_dir:  /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/labeled_example_data
